# T20 — Simulation Patterns Deep Dive

Explore all 6 simulation pattern modules: clickstream, financial streams, IoT telemetry, operational logs, SCD2 file drops, and workflow state machines.

**What you'll learn:**
- Configure and run each simulation pattern
- Understand the config/result/simulator pattern
- Generate realistic operational data for pipeline testing

## 1. Clickstream Simulation

In [1]:
from sqllocks_spindle.simulation.clickstream_patterns import ClickstreamConfig, ClickstreamSimulator

config = ClickstreamConfig(
    users=100,
    avg_sessions_per_user=5,
    avg_pages_per_session=8,
    bot_fraction=0.05,
    seed=42,
)
sim = ClickstreamSimulator(config)
result = sim.run()

print(f"Clickstream: {len(result.page_views)} page views, {result.stats['unique_users']} users, {result.stats['total_sessions']} sessions")
result.page_views.head()

Clickstream: 4025 page views, 126 users, 534 sessions


,view_id,session_id,user_id,page_url,timestamp,time_on_page_seconds,referrer_url
0,c32ca3d5-81f7-4eb9-baa6-d2a2f537ec95,51bedb8c-e48c-4bf4-ade8-ea0df954adf9,user_000088,/account,2026-03-17 00:05:12.443549+00:00,17.51,direct
1,4a70d44f-f710-4972-9574-4ee3b1842c55,97f92f58-9f6b-4d4b-9f54-c52839529c55,user_000018,/blog,2026-03-17 00:38:16.318896+00:00,9.42,facebook
2,dc25ddff-b142-4543-b561-641454e97b71,97f92f58-9f6b-4d4b-9f54-c52839529c55,user_000018,/blog/post-244,2026-03-17 00:38:25.741309524+00:00,45.46,/blog
3,d49f8557-7ce0-499f-9670-07289082ce15,97f92f58-9f6b-4d4b-9f54-c52839529c55,user_000018,/search,2026-03-17 00:39:11.203147825+00:00,102.95,/blog/post-244
4,a0d4da10-d6fa-4fa5-a057-9115520f32a2,97f92f58-9f6b-4d4b-9f54-c52839529c55,user_000018,/products,2026-03-17 00:40:54.153250303+00:00,5.00,/search


## 2. Financial Stream Simulation

In [2]:
from sqllocks_spindle import Spindle, FinancialDomain
from sqllocks_spindle.simulation.financial_patterns import FinancialStreamConfig, FinancialStreamSimulator

# Generate base financial data
baseline = Spindle().generate(domain=FinancialDomain(), scale="small", seed=42)

config = FinancialStreamConfig(
    reversal_probability=0.03,
    fraud_burst_probability=0.01,
    seed=42,
)
sim = FinancialStreamSimulator(
    transactions_df=baseline.tables["transaction"],
    accounts_df=baseline.tables["account"],
    config=config,
)
result = sim.run()

print(f"Financial: {len(result.transactions)} transactions")
print(f"  Reversals: {result.stats['reversal_count']}")
print(f"  Fraud events: {result.stats['fraud_event_count']}")
result.transactions.head()

Financial: 10292 transactions
  Reversals: 292
  Fraud events: 0


,transaction_id,account_id,category_id,transaction_type,amount,transaction_date,channel,merchant_name,status,reversal_id,original_transaction_id,reversal_reason,reversed_at
0,1.0,7,1.0,Payment,41.141261,2023-02-28 13:51:35.696674698,Phone,Chipotle,Completed,NaN,NaN,NaN,NaT
1,2.0,4,4.0,Deposit,227.025456,2022-12-16 02:06:26.058998096,ATM,Lyft,Completed,NaN,NaN,NaN,NaT
2,3.0,11,1.0,Withdrawal,12.287186,2024-05-13 03:41:26.179915942,Mobile,Starbucks,Completed,NaN,NaN,NaN,NaT
3,4.0,7,9.0,Deposit,45.291048,2022-11-05 09:00:14.697103224,Phone,Whole Foods,Completed,NaN,NaN,NaN,NaT
4,5.0,1,11.0,Deposit,129.816960,2023-09-05 14:19:44.191903758,Wire,NaN,Completed,NaN,NaN,NaN,NaT


## 3. IoT Telemetry Simulation

In [3]:
from sqllocks_spindle import Spindle, IoTDomain
from sqllocks_spindle.simulation.iot_patterns import IoTTelemetryConfig, IoTTelemetrySimulator

# Generate base IoT data
baseline = Spindle().generate(domain=IoTDomain(), scale="small", seed=42)

config = IoTTelemetryConfig(
    fleet_size=20,
    seed=42,
)
sim = IoTTelemetrySimulator(
    readings_df=baseline.tables["reading"],
    devices_df=baseline.tables["device"],
    config=config,
)
result = sim.run()

print(f"IoT: {result.stats['total_readings']} readings from {len(result.fleet_status)} devices")
print(f"  Alerts: {result.stats['total_alerts']}")
result.readings.head()

IoT: 25000 readings from 500 devices
  Alerts: 16580


,reading_id,sensor_id,reading_value,reading_timestamp,quality_flag
0,1,61,24.962267,2025-07-15 05:52:51.037233313,Good
1,2,3,25.609141,2025-06-20 07:16:15.108054346,Bad
2,3,1,24.594949,2023-01-27 04:12:31.950822615,Good
3,4,4,25.360784,2024-09-10 07:39:47.438203114,Good
4,5,1,23.774225,2024-09-01 10:02:50.271389082,Good


## 4. Operational Log Simulation

In [4]:
from sqllocks_spindle.simulation.operational_log_patterns import OperationalLogConfig, OperationalLogSimulator

config = OperationalLogConfig(
    events_per_hour=100.0,
    outage_probability=0.02,
    seed=42,
)
sim = OperationalLogSimulator(config)
result = sim.run()

print(f"Ops logs: {result.stats['total_events']} entries")
print(f"  Errors: {result.stats['total_errors']}")
result.logs.head()

Ops logs: 12211 entries
  Errors: 1037


,log_id,timestamp,service,tier,level,method,endpoint,status_code,latency_ms,message,trace_id,span_id,is_spike,is_outage
0,95d35f97-7f84-4bdd-b21e-981fdbc560b9,2024-01-01 00:00:14.018927,order-service,core,INFO,GET,/api/v1/orders,200,40.01,Handled request in 40.01ms,c2e904b7-0325-4638-87a8-715a987c50aa,2e959899-6afd-40,False,False
1,ed3fc835-a8f0-4e1f-86aa-3b571d057b75,2024-01-01 00:00:19.703192,auth-service,middleware,INFO,DELETE,/readyz,200,58.11,Handled request in 58.11ms,62c9b0b3-f7c9-454b-a0a1-ec15d98c9746,0d5b1841-c294-42,False,False
2,3bd630dd-76ad-4355-b6b0-4d170ff026a3,2024-01-01 00:00:21.809222,payment-service,core,INFO,GET,/api/v1/orders,200,34.45,Handled request in 34.45ms,08c47159-88b6-4c98-94f8-446a5f5da5b0,ca3d869d-9d7e-4f,False,False
3,ee92e0d7-080d-45d0-99c0-3302e705014c,2024-01-01 00:00:57.298691,payment-service,core,INFO,POST,/readyz,200,95.54,Handled request in 95.54ms,b9c21de6-50ec-4e8c-8f40-dade97812895,3c19ce22-c7ee-46,False,False
4,3b848219-0bc0-4832-a161-94b70db98ad6,2024-01-01 00:01:11.159203,api-gateway,edge,INFO,PUT,/readyz,200,23.83,Handled request in 23.83ms,7ac62b4d-a574-4810-88e5-7a7fbe920d30,c93a9405-57da-46,False,False


## 5. SCD2 File Drop Simulation

In [5]:
from sqllocks_spindle.simulation.scd2_file_drops import SCD2FileDropConfig, SCD2FileDropSimulator
from sqllocks_spindle import Spindle, RetailDomain

baseline = Spindle().generate(domain=RetailDomain(), scale="small", seed=42)

config = SCD2FileDropConfig(
    domain="retail",
    business_key_column="customer_id",
    num_delta_days=5,
    daily_change_rate=0.05,
    daily_new_rate=0.02,
    seed=42,
)
sim = SCD2FileDropSimulator(tables={"customer": baseline.tables["customer"]}, config=config)
result = sim.run()

print(f"SCD2 file drops: {len(result.delta_paths)} delta files + 1 initial load")
print(f"  Initial rows: {result.stats['initial_rows']}")
print(f"  Total new: {result.stats['total_new']}")
print(f"  Total updates: {result.stats['total_updates']}")
print(f"  Total delta records: {result.stats['total_deltas']}")

SCD2 file drops: 5 delta files + 1 initial load
  Initial rows: 1000
  Total new: 102
  Total updates: 260
  Total delta records: 622


## 6. Workflow State Machine

In [6]:
from sqllocks_spindle.simulation.state_machine import WorkflowConfig, WorkflowSimulator, get_preset_workflow

# Use built-in order fulfillment preset
states, transitions = get_preset_workflow("order_fulfillment")
config = WorkflowConfig(states=states, transitions=transitions, entity_count=200, seed=42)
sim = WorkflowSimulator(config)
result = sim.run()

print(f"Workflow: {len(result.events)} state transitions for {result.stats['total_entities']} entities")
print(f"Terminal state distribution: {result.state_distribution}")
result.events.head(10)

Workflow: 528 state transitions for 200 entities
Terminal state distribution: {'delivered': 138, 'cancelled': 38, 'returned': 15, 'confirmed': 4, 'shipped': 3, 'created': 2}


,event_id,entity_id,from_state,to_state,transitioned_at,dwell_hours,is_anomaly,anomaly_type
0,f7cc99af-2768-4d9f-a078-dbfe200d105d,entity_000000,created,confirmed,2024-01-01 03:14:39.258265,2.4703,False,
1,ba339e99-be46-4115-9f54-df31945dc55b,entity_000000,confirmed,shipped,2024-01-02 04:00:40.610973,24.7670,False,
2,ee5dd647-a91c-475b-817e-3c7f253d8366,entity_000000,shipped,delivered,2024-01-04 07:32:17.615631,51.5269,False,
3,431e94e3-bfb5-4012-91bb-7eea9043bf0d,entity_000001,created,cancelled,2024-01-01 01:42:32.293391,1.3382,False,
4,21f18eef-404b-4287-94a0-2f2b51d66dab,entity_000002,created,confirmed,2024-01-01 01:57:50.302435,1.5206,False,
5,8d420a57-dcdd-4bb8-990e-1d4624c0e4b3,entity_000002,confirmed,shipped,2024-01-02 00:51:17.275382,22.8908,False,
6,86a8c4c1-a847-4449-971f-ac034c37627a,entity_000002,shipped,delivered,2024-01-04 21:08:45.928131,68.2913,False,
7,48932700-f9ae-497f-a826-b0ac95a02d47,entity_000003,created,confirmed,2024-01-01 02:57:39.979905,2.1827,False,
8,c1350b3b-e26a-42d4-b821-f48f96236f95,entity_000003,confirmed,shipped,2024-01-02 15:48:39.568084,36.8499,False,
9,2bc34087-d113-4752-88fd-14dd449f47a2,entity_000003,shipped,returned,2024-01-06 10:30:54.622503,90.7042,False,


## Pattern Summary

| Pattern | Module | Config Class | Use Case |
| --- | --- | --- | --- |
| Clickstream | `simulation.clickstream_patterns` | `ClickstreamConfig` | Web analytics, funnel testing |
| Financial | `simulation.financial_patterns` | `FinancialStreamConfig` | Transaction fraud, reversals |
| IoT | `simulation.iot_patterns` | `IoTTelemetryConfig` | Sensor data, anomaly detection |
| Ops Logs | `simulation.operational_log_patterns` | `OperationalLogConfig` | Observability, incident response |
| SCD2 | `simulation.scd2_file_drops` | `SCD2FileDropConfig` | Slowly-changing dimensions |
| Workflow | `simulation.state_machine` | `WorkflowConfig` | Entity lifecycle, state transitions |